# 🔴 Red Laser Boundary Detector
**Zapdos Labs — Forklift Safety Zone Masking**

Pipeline:
1. Clone repo & load data
2. HSV red segmentation
3. Morphological denoising
4. Convex hull boundary fitting
5. Output clean binary mask video

## 1. Setup

In [ ]:
# Install deps
!pip install opencv-python-headless numpy matplotlib -q

# Clone the repo
!git clone https://github.com/zapdos-labs/technical-interview.git

import os, glob
data_dir = '/content/technical-interview/data'
clips = sorted(glob.glob(f'{data_dir}/**/*.mp4', recursive=True) +
               glob.glob(f'{data_dir}/**/*.mov', recursive=True) +
               glob.glob(f'{data_dir}/**/*.avi', recursive=True))
print(f'Found {len(clips)} video clips:')
for c in clips:
    print(' ', c)

## 2. Core Pipeline

In [ ]:
import cv2
import numpy as np

# ─────────────────────────────────────────────
# TUNABLE PARAMETERS
# ─────────────────────────────────────────────
class Config:
    # HSV red range — red wraps around hue 0/180
    RED_LOW_1  = np.array([0,   120,  80],  dtype=np.uint8)
    RED_HIGH_1 = np.array([10,  255, 255],  dtype=np.uint8)
    RED_LOW_2  = np.array([165, 120,  80],  dtype=np.uint8)
    RED_HIGH_2 = np.array([180, 255, 255],  dtype=np.uint8)

    # Morphology kernels
    OPEN_KERNEL   = 3    # removes dust / speckle
    CLOSE_KERNEL  = 21   # fills gaps between laser line segments
    DILATE_KERNEL = 5    # slightly thickens after closing

    # Contour filtering
    MIN_AREA = 500       # px² — ignore tiny blobs

    # Temporal smoothing (alpha for EMA on mask)
    TEMPORAL_ALPHA = 0.6  # 0=no smoothing, 1=no memory


def red_mask(frame_bgr):
    """Binary mask of red pixels via HSV thresholding."""
    hsv = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2HSV)
    m1  = cv2.inRange(hsv, Config.RED_LOW_1,  Config.RED_HIGH_1)
    m2  = cv2.inRange(hsv, Config.RED_LOW_2,  Config.RED_HIGH_2)
    return cv2.bitwise_or(m1, m2)


def denoise(mask):
    """Remove speckle, then fill gaps along laser lines."""
    k_open  = cv2.getStructuringElement(
        cv2.MORPH_ELLIPSE, (Config.OPEN_KERNEL,  Config.OPEN_KERNEL))
    k_close = cv2.getStructuringElement(
        cv2.MORPH_RECT,    (Config.CLOSE_KERNEL, Config.CLOSE_KERNEL))
    k_dil   = cv2.getStructuringElement(
        cv2.MORPH_ELLIPSE, (Config.DILATE_KERNEL, Config.DILATE_KERNEL))
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN,  k_open)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, k_close)
    mask = cv2.dilate(mask, k_dil)
    return mask


def fit_boundary(mask):
    """Convex hull over all laser blobs → clean filled white region."""
    out = np.zeros_like(mask)
    contours, _ = cv2.findContours(
        mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return out
    good = [c for c in contours if cv2.contourArea(c) >= Config.MIN_AREA]
    if not good:
        return out
    all_pts = np.vstack(good)
    hull = cv2.convexHull(all_pts)
    cv2.fillPoly(out, [hull], 255)
    return out


def process_frame(frame_bgr):
    """Single frame → binary mask."""
    raw   = red_mask(frame_bgr)
    clean = denoise(raw)
    final = fit_boundary(clean)
    return final


print('Pipeline defined ✓')

## 3. Visualise Pipeline Stages on One Frame

In [ ]:
import matplotlib.pyplot as plt

def show_stages(video_path, frame_idx=30):
    cap = cv2.VideoCapture(video_path)
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
    ret, frame = cap.read()
    cap.release()
    if not ret:
        cap = cv2.VideoCapture(video_path)
        ret, frame = cap.read()
        cap.release()
    assert ret, 'Could not read frame'

    raw   = red_mask(frame)
    clean = denoise(raw)
    final = fit_boundary(clean)

    overlay = frame.copy()
    overlay[final > 0] = (0, 200, 0)
    blended = cv2.addWeighted(frame, 0.55, overlay, 0.45, 0)

    titles = ['Original', 'Raw Red Mask', 'After Denoise', 'Final Mask', 'Overlay']
    imgs   = [
        cv2.cvtColor(frame,   cv2.COLOR_BGR2RGB),
        raw, clean, final,
        cv2.cvtColor(blended, cv2.COLOR_BGR2RGB)
    ]

    fig, axes = plt.subplots(1, 5, figsize=(22, 4))
    for ax, img, title in zip(axes, imgs, titles):
        cmap = None if img.ndim == 3 else 'gray'
        ax.imshow(img, cmap=cmap)
        ax.set_title(title, fontsize=11)
        ax.axis('off')
    plt.suptitle(os.path.basename(video_path), fontsize=13, y=1.02)
    plt.tight_layout()
    plt.show()

# Show stages for first clip
if clips:
    show_stages(clips[0])
else:
    print('No clips found — check data_dir path')

## 4. (Optional) Tune HSV Thresholds Interactively

In [ ]:
# Run this cell to try different HSV ranges on a specific frame
# Change the values below and re-run to see effect immediately

CLIP_INDEX = 0      # which clip to preview
FRAME_INDEX = 30    # which frame

# Try widening saturation (second value) if laser is faint
# Try widening hue range if some laser frames are missed
TEST_LOW_1  = [0,   100, 60]
TEST_HIGH_1 = [12,  255, 255]
TEST_LOW_2  = [160, 100, 60]
TEST_HIGH_2 = [180, 255, 255]

cap = cv2.VideoCapture(clips[CLIP_INDEX])
cap.set(cv2.CAP_PROP_POS_FRAMES, FRAME_INDEX)
ret, frame = cap.read()
cap.release()

hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
m1  = cv2.inRange(hsv, np.array(TEST_LOW_1),  np.array(TEST_HIGH_1))
m2  = cv2.inRange(hsv, np.array(TEST_LOW_2),  np.array(TEST_HIGH_2))
test_mask = cv2.bitwise_or(m1, m2)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
ax1.set_title('Original')
ax1.axis('off')
ax2.imshow(test_mask, cmap='gray')
ax2.set_title('Test HSV Mask')
ax2.axis('off')
plt.tight_layout()
plt.show()

# If happy, update Config:
# Config.RED_LOW_1  = np.array(TEST_LOW_1,  dtype=np.uint8)
# Config.RED_HIGH_1 = np.array(TEST_HIGH_1, dtype=np.uint8)
# Config.RED_LOW_2  = np.array(TEST_LOW_2,  dtype=np.uint8)
# Config.RED_HIGH_2 = np.array(TEST_HIGH_2, dtype=np.uint8)

## 5. Process All Clips → Save Mask Videos

In [ ]:
import time

output_dir = '/content/output_masks'
os.makedirs(output_dir, exist_ok=True)

def process_video(input_path, output_path):
    cap = cv2.VideoCapture(input_path)
    fps    = cap.get(cv2.CAP_PROP_FPS) or 30
    width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total  = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    writer = cv2.VideoWriter(output_path, fourcc, fps, (width, height), isColor=False)

    prev_mask = None
    frame_count = 0
    t0 = time.time()

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        mask = process_frame(frame)

        # Temporal smoothing: EMA blend with previous mask
        if prev_mask is not None:
            blended = cv2.addWeighted(
                mask.astype(np.float32), Config.TEMPORAL_ALPHA,
                prev_mask.astype(np.float32), 1 - Config.TEMPORAL_ALPHA, 0)
            mask = (blended > 127).astype(np.uint8) * 255
        prev_mask = mask.copy()

        writer.write(mask)
        frame_count += 1

    cap.release()
    writer.release()

    elapsed = time.time() - t0
    achieved_fps = frame_count / elapsed if elapsed > 0 else 0
    return frame_count, achieved_fps


print(f'Processing {len(clips)} clips...\n')
results = []
for clip in clips:
    name   = os.path.splitext(os.path.basename(clip))[0]
    out    = f'{output_dir}/{name}_mask.mp4'
    n, fps = process_video(clip, out)
    results.append((name, n, fps, out))
    status = '✓' if fps >= 2 else '⚠ SLOW'
    print(f'  {status}  {name:30s}  {n:4d} frames  {fps:.1f} FPS  →  {out}')

print(f'\nAll done. Outputs in {output_dir}/')

## 6. Visual QA — Compare Input vs Mask Side by Side

In [ ]:
def qa_strip(input_path, mask_path, n_frames=6):
    """Show n evenly-spaced frame pairs: input | mask."""
    cap_in   = cv2.VideoCapture(input_path)
    cap_mask = cv2.VideoCapture(mask_path)
    total = int(cap_in.get(cv2.CAP_PROP_FRAME_COUNT))
    idxs  = np.linspace(0, total - 1, n_frames, dtype=int)

    fig, axes = plt.subplots(2, n_frames, figsize=(4 * n_frames, 5))
    for col, idx in enumerate(idxs):
        for cap, row, cmap, title in [
            (cap_in,   0, None,   'Input'),
            (cap_mask, 1, 'gray', 'Mask')
        ]:
            cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
            ret, f = cap.read()
            if ret:
                img = cv2.cvtColor(f, cv2.COLOR_BGR2RGB) if cmap is None else \
                      cv2.cvtColor(f, cv2.COLOR_BGR2GRAY)
                axes[row][col].imshow(img, cmap=cmap)
            axes[row][col].axis('off')
            if col == 0:
                axes[row][col].set_ylabel(title, fontsize=12)
    cap_in.release()
    cap_mask.release()
    plt.suptitle(os.path.basename(input_path), fontsize=13)
    plt.tight_layout()
    plt.show()

# Run QA on all processed clips
for clip in clips:
    name = os.path.splitext(os.path.basename(clip))[0]
    mask_path = f'{output_dir}/{name}_mask.mp4'
    if os.path.exists(mask_path):
        qa_strip(clip, mask_path)

## 7. Download All Outputs as ZIP

In [ ]:
import shutil
from google.colab import files

zip_path = '/content/laser_masks.zip'
shutil.make_archive('/content/laser_masks', 'zip', output_dir)
files.download(zip_path)
print('Download started!')

---
## Tuning Cheatsheet

| Problem | Fix |
|---|---|
| Laser not detected | Widen HSV ranges (`RED_LOW_1` lower sat/val) |
| Floor glare bleeding in | Raise `MIN_AREA`, narrow hue range |
| Gaps in the hull | Increase `CLOSE_KERNEL` (try 25–35) |
| Flickery output | Increase `TEMPORAL_ALPHA` toward 0.4 |
| Too slow | Reduce resolution: add `frame = cv2.resize(frame, (640, 360))` |
| Perspective warp | Convex hull handles this — if shape is non-convex, use `minAreaRect` instead |